In [7]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

np.random.seed(42)
n_samples = 3000

models_list = np.random.choice(['Gentra', 'Malibu', 'Tracker', 'Cobalt', 'Nexia 3'], size=n_samples)
years = np.random.randint(2015, 2027, size=n_samples)
probeg = (2026 - years) * np.random.randint(15000, 25000, size=n_samples) + np.random.randint(0, 10000, size=n_samples)
probeg = np.clip(probeg, 0, 400000)

car_type = np.where((models_list == 'Tracker'), 1, 0)
car_condition = np.random.choice([0, 1], size=n_samples, p=[0.3, 0.7])

base_prices = {'Gentra': 11000, 'Malibu': 24000, 'Tracker': 19000, 'Cobalt': 10500, 'Nexia 3': 8000}
prices = np.array([base_prices[m] for m in models_list])

prices = prices + (years - 2015) * 600
prices = prices - (probeg / 1000) * 15
prices = prices + (car_condition * 1500)
prices = prices + np.random.randint(-500, 500, size=n_samples)

df_fresh = pd.DataFrame({
    'model_name': models_list,
    'year': years,
    'probeg': probeg,
    'car_type_dl': car_type,
    'condition_nlp': car_condition,
    'price': prices
})

df_encoded = pd.get_dummies(df_fresh, columns=['model_name'], drop_first=False)

X = df_encoded.drop(columns=['price']).copy()
y = df_encoded['price'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("1. Yakuniy ML Narx modeli o'qitilmoqda...")
ml_price_model = RandomForestRegressor(n_estimators=100, random_state=42)
ml_price_model.fit(X_train, y_train)

y_pred = ml_price_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\n✅ Model TAYYOR!")
print(f"--- O'rtacha xatolik (MAE): ${mae:.2f}")
print(f"--- Model aniqligi (R2 Score): {r2*100:.2f}%")

joblib.dump(ml_price_model, 'auto_price_model.joblib')
joblib.dump(X.columns.tolist(), 'auto_price_features.joblib')



1. Yakuniy ML Narx modeli o'qitilmoqda...

✅ Model TAYYOR!
--- O'rtacha xatolik (MAE): $288.55
--- Model aniqligi (R2 Score): 99.72%


['auto_price_features.joblib']